# Sports Card Extractor (Enhanced)

Extract structured data from sports card images using OpenAI's vision API with **dual-image support** (front + back).

## Features
- 📁 **Local file upload** - Use images from your computer
- 🔄 **Front + Back processing** - Extract from both sides simultaneously
- 🎯 **Comprehensive extraction** - Brand, year, player, card #, serial, auto, parallels, visual descriptors
- 🔍 **eBay search generation** - Ready-to-use search URLs

## 1. Setup & Install

In [ ]:
!pip install openai python-dotenv pydantic pandas ipywidgets --quiet

In [ ]:
import os
import base64
import webbrowser
from pathlib import Path
from urllib.parse import quote
from typing import Optional, List

import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown, HTML

# For local file uploads
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Note: ipywidgets not available. Using file path input instead.")

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

## 2. Data Model (Pydantic)

Comprehensive schema for sports cards based on industry metadata standards:
- **Identification**: Player, Team, Position, Sport
- **Card Details**: Brand, Set, Year, Card Number
- **Rarity Indicators**: Serial Number, Parallel Type, Insert Name
- **Special Features**: Autograph, Memorabilia/Relic, Rookie Card
- **Grading**: Company, Grade
- **Visual Description**: Colors, patterns, finishes for comprehensive search

In [ ]:
class SportsCard(BaseModel):
    """Comprehensive structured representation of a sports card."""

    # === CORE IDENTIFICATION ===
    player_name: str = Field(
        description="Full name of the player (e.g. 'Aaron Judge', 'Patrick Mahomes II')"
    )
    team: Optional[str] = Field(
        default=None,
        description="Team name shown on card (e.g. 'New York Yankees', 'Kansas City Chiefs')"
    )
    position: Optional[str] = Field(
        default=None,
        description="Player position (e.g. 'OF', 'QB', 'PG', 'C', 'WR')"
    )
    sport: Optional[str] = Field(
        default=None,
        description="Sport: Baseball, Football, Basketball, Hockey, Soccer"
    )
    
    # === CARD DETAILS (often on back) ===
    manufacturer: Optional[str] = Field(
        default=None,
        description="Card manufacturer/brand (e.g. 'Topps', 'Panini', 'Upper Deck', 'Bowman', 'Donruss', 'Fleer')"
    )
    product_set: Optional[str] = Field(
        default=None,
        description="Product/Set name (e.g. 'Topps Chrome', 'Prizm', 'Finest', 'Optic', 'Mosaic', 'Select')"
    )
    year: Optional[int] = Field(
        default=None,
        description="Year from copyright/fine print on back of card (e.g. 2024, 2025)"
    )
    card_number: Optional[str] = Field(
        default=None,
        description="Card number from back of card (e.g. '51', '150', 'RC-25')"
    )
    
    # === RARITY & VARIANTS ===
    serial_number: Optional[str] = Field(
        default=None,
        description="Serial numbering if present (e.g. '25/99', '1/1', '517/2000'). Found on front or back."
    )
    parallel_type: Optional[str] = Field(
        default=None,
        description="Parallel/refractor type (e.g. 'Silver Prizm', 'Gold Refractor', 'Blue Wave', 'Cracked Ice', 'Superfractor', 'X-Fractor', 'Aqua Lava', 'Snakeskin')"
    )
    insert_name: Optional[str] = Field(
        default=None,
        description="Insert set name if not a base card (e.g. 'League Leaders', 'Award Winners', 'Rookie Debut')"
    )
    
    # === SPECIAL FEATURES ===
    is_rookie_card: Optional[bool] = Field(
        default=None,
        description="True if marked as rookie card (RC logo, 'Rookie' text, or 1st Bowman)"
    )
    is_autograph: Optional[bool] = Field(
        default=None,
        description="True if card contains an autograph (usually on front)"
    )
    is_memorabilia: Optional[bool] = Field(
        default=None,
        description="True if card contains jersey patch, relic, or game-used material"
    )
    
    # === GRADING (if slabbed) ===
    grading_company: Optional[str] = Field(
        default=None,
        description="Grading company if card is slabbed (e.g. 'PSA', 'BGS', 'SGC', 'CGC')"
    )
    grade: Optional[str] = Field(
        default=None,
        description="Grade value (e.g. '10', '9.5', 'GEM MT 10', 'PRISTINE 10')"
    )
    
    # === VISUAL DESCRIPTION (for comprehensive search) ===
    visual_description: Optional[str] = Field(
        default=None,
        description="Visual characteristics: colors, patterns, finishes, special effects (e.g. 'purple border, holographic shimmer, stained glass pattern, rainbow refractor finish, cracked ice texture')"
    )
    
    # === ADDITIONAL IDENTIFIERS ===
    additional_text: Optional[str] = Field(
        default=None,
        description="Any other identifying text on the card (e.g. 'League Leaders', 'All-Star', 'MVP', 'Photo Variation', 'Short Print SP', 'Super Short Print SSP')"
    )

    @property
    def display_name(self) -> str:
        parts = []
        if self.year:
            parts.append(str(self.year))
        if self.manufacturer:
            parts.append(self.manufacturer)
        if self.product_set:
            parts.append(self.product_set)
        parts.append(self.player_name)
        if self.parallel_type:
            parts.append(f"- {self.parallel_type}")
        if self.is_rookie_card:
            parts.append("RC")
        if self.serial_number:
            parts.append(f"[{self.serial_number}]")
        return " ".join(parts)

    @property
    def ebay_search_query(self) -> str:
        """Build optimized eBay search string following best practices."""
        tokens = []
        
        # Year first (most important filter)
        if self.year:
            tokens.append(str(self.year))
        
        # Brand/manufacturer
        if self.manufacturer:
            tokens.append(self.manufacturer)
        
        # Product set
        if self.product_set:
            tokens.append(self.product_set)
        
        # Player name (core)
        tokens.append(self.player_name)
        
        # Card number
        if self.card_number:
            tokens.append(f"#{self.card_number}")
        
        # Parallel type (important for pricing)
        if self.parallel_type:
            tokens.append(self.parallel_type)
        
        # Serial numbering (shows rarity)
        if self.serial_number:
            # Extract just the print run (e.g., /99 from 25/99)
            parts = self.serial_number.split('/')
            if len(parts) == 2:
                tokens.append(f"/{parts[1]}")
        
        # Special features
        if self.is_rookie_card:
            tokens.append("RC")
        if self.is_autograph:
            tokens.append("Auto")
        if self.is_memorabilia:
            tokens.append("Relic")
        
        # Grade if present
        if self.grade:
            company = self.grading_company or "PSA"
            tokens.append(f"{company} {self.grade}")
        
        return " ".join(tokens)

    @property
    def ebay_url(self) -> str:
        return f"https://www.ebay.com/sch/i.html?_nkw={quote(self.ebay_search_query)}"

    def to_markdown_table(self) -> str:
        """Render card data as a markdown table."""
        lines = ["| **Field** | **Value** |", "| :-- | :-- |"]
        for field_name, value in self.model_dump().items():
            if value is not None and value != [] and value != False:
                label = field_name.replace("_", " ").title()
                if isinstance(value, bool):
                    value = "✅ Yes"
                if isinstance(value, list):
                    value = ", ".join(str(v) for v in value)
                lines.append(f"| {label} | {value} |")
        return "\n".join(lines)

    def _repr_markdown_(self):
        return f"### {self.display_name}\n\n{self.to_markdown_table()}"

## 3. Image Handling

Support for:
- Local files (upload or path)
- URLs
- Front + Back dual image processing

In [ ]:
def encode_image_to_base64(image_source: str) -> dict:
    """Convert an image source (URL or local file) to API-ready format."""
    if image_source.startswith(("http://", "https://")):
        return {
            "type": "image_url",
            "image_url": {"url": image_source},
        }
    else:
        # Local file - read and encode
        with open(image_source, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")
        ext = os.path.splitext(image_source)[1].lower().lstrip(".")
        mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp", "gif": "gif"}.get(ext, "jpeg")
        return {
            "type": "image_url",
            "image_url": {"url": f"data:image/{mime};base64,{b64}"},
        }


def encode_uploaded_file(uploaded_file) -> dict:
    """Convert an ipywidgets FileUpload content to API-ready format."""
    content = uploaded_file['content']
    name = uploaded_file['name']
    ext = os.path.splitext(name)[1].lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp", "gif": "gif"}.get(ext, "jpeg")
    b64 = base64.b64encode(content).decode("utf-8")
    return {
        "type": "image_url",
        "image_url": {"url": f"data:image/{mime};base64,{b64}"},
    }

## 4. Extraction Function (Dual Image Support)

Processes **front and back** of card together for complete extraction.

In [ ]:
SYSTEM_PROMPT = """
You are an expert sports card analyst and collector. You will receive images of a sports card 
(front and/or back). Extract all structured information following these priorities:

EXTRACTION PRIORITY ORDER:
1. BRAND/MANUFACTURER - Look for Topps, Panini, Upper Deck, Bowman, Donruss, Fleer logos
2. PRODUCT SET - Identify the specific product (Chrome, Prizm, Finest, Optic, Mosaic, Select, etc.)
3. YEAR - Find in fine print/copyright on BACK of card (e.g., "© 2025 Topps")
4. PLAYER NAME - Full name as shown on card
5. CARD NUMBER - Usually on BACK, sometimes front (e.g., "#51", "RC-25")
6. SERIAL NUMBER - Limited print notation like "25/99" or "1/1" (front or back)
7. AUTOGRAPH - Look for signature, usually on FRONT
8. PARALLEL TYPE - Identify color variants, refractors, special finishes:
   - Topps: Refractor, Gold, Blue, Green, Purple, Rainbow Foil, X-Fractor, Superfractor
   - Panini: Silver Prizm, Blue Prizm, Green Wave, Cracked Ice, Snakeskin, Disco, Lazer
   - Upper Deck: Exclusives, Clear Cut, High Gloss, Outburst
9. VISUAL DESCRIPTION - Describe colors, patterns, special effects you observe
10. OTHER IDENTIFIERS - Insert names, SP/SSP, Photo Variation, Award mentions

IMPORTANT:
- Use null for any field you cannot confidently determine
- The BACK of the card is crucial for: Year, Card Number, Copyright info
- The FRONT is crucial for: Player image, Autograph, Parallel appearance, Serial number
- Be precise with serial numbers (format: XX/YY)
- Visual description should help with search (colors, effects, textures)
"""


def extract_card(
    front_image: str = None,
    back_image: str = None,
    front_upload: dict = None,
    back_upload: dict = None,
) -> SportsCard:
    """
    Extract card data from front and/or back images.
    
    Args:
        front_image: URL or local path to front of card
        back_image: URL or local path to back of card
        front_upload: Dict from ipywidgets FileUpload (front)
        back_upload: Dict from ipywidgets FileUpload (back)
    """
    # Build image content list
    content_parts = [{"type": "text", "text": "Extract all structured data from this sports card. I'm providing the front and/or back images."}]
    
    # Add front image
    if front_upload:
        content_parts.append({"type": "text", "text": "FRONT OF CARD:"})
        content_parts.append(encode_uploaded_file(front_upload))
    elif front_image:
        content_parts.append({"type": "text", "text": "FRONT OF CARD:"})
        content_parts.append(encode_image_to_base64(front_image))
    
    # Add back image
    if back_upload:
        content_parts.append({"type": "text", "text": "BACK OF CARD:"})
        content_parts.append(encode_uploaded_file(back_upload))
    elif back_image:
        content_parts.append({"type": "text", "text": "BACK OF CARD:"})
        content_parts.append(encode_image_to_base64(back_image))
    
    if len(content_parts) == 1:
        raise ValueError("Must provide at least one image (front or back)")
    
    # Call API with structured output
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content_parts},
        ],
        response_format=SportsCard,
        temperature=0.1,
    )

    card = completion.choices[0].message.parsed

    if card is None:
        refusal = completion.choices[0].message.refusal
        raise ValueError(f"Extraction failed: {refusal or 'No structured output returned'}")

    return card

## 5. File Upload Widget

Upload card images directly from your computer.

In [ ]:
if WIDGETS_AVAILABLE:
    # Create upload widgets
    front_upload = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Front of Card'
    )
    
    back_upload = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Back of Card'
    )
    
    extract_button = widgets.Button(
        description='Extract Card Data',
        button_style='success',
        icon='search'
    )
    
    output = widgets.Output()
    
    def on_extract_click(b):
        with output:
            clear_output()
            
            front_data = None
            back_data = None
            
            if front_upload.value:
                front_data = list(front_upload.value.values())[0]
                print(f"✓ Front image: {front_data['name']}")
            
            if back_upload.value:
                back_data = list(back_upload.value.values())[0]
                print(f"✓ Back image: {back_data['name']}")
            
            if not front_data and not back_data:
                print("⚠️ Please upload at least one image (front or back)")
                return
            
            print("\n🔍 Extracting card data...\n")
            
            try:
                card = extract_card(front_upload=front_data, back_upload=back_data)
                display(card)
                print(f"\n🔗 eBay Search: {card.ebay_url}")
            except Exception as e:
                print(f"❌ Error: {e}")
    
    extract_button.on_click(on_extract_click)
    
    print("📤 Upload your card images below:")
    print("   - Front of card (recommended)")
    print("   - Back of card (for year, card number, fine print)")
    print("")
    display(widgets.VBox([
        widgets.HBox([widgets.Label("Front:"), front_upload]),
        widgets.HBox([widgets.Label("Back: "), back_upload]),
        extract_button,
        output
    ]))
else:
    print("Widget upload not available. Use the file path method in the next cell.")

## 6. Alternative: File Path Input

If widgets aren't available, use direct file paths or URLs.

In [ ]:
# Option 1: Local file paths
# FRONT_IMAGE = "/path/to/your/card_front.jpg"
# BACK_IMAGE = "/path/to/your/card_back.jpg"

# Option 2: URLs
FRONT_IMAGE = "https://i.ebayimg.com/images/g/fBQAAOSwbRVl8o0d/s-l1600.webp"  # Example card front
BACK_IMAGE = None  # Set to URL or path if you have back image

# Preview
if FRONT_IMAGE:
    print("Front of card:")
    if FRONT_IMAGE.startswith("http"):
        display(Image(url=FRONT_IMAGE, width=300))
    else:
        display(Image(filename=FRONT_IMAGE, width=300))

In [ ]:
# Extract from file paths/URLs
card = extract_card(front_image=FRONT_IMAGE, back_image=BACK_IMAGE)

# Display results
card

## 7. Inspect Extracted Data

In [ ]:
# Access fields directly
print("=== EXTRACTED DATA ===")
print(f"Player: {card.player_name}")
print(f"Team: {card.team}")
print(f"Sport: {card.sport}")
print(f"Year: {card.year}")
print(f"Brand: {card.manufacturer}")
print(f"Set: {card.product_set}")
print(f"Card #: {card.card_number}")
print(f"Serial: {card.serial_number}")
print(f"Parallel: {card.parallel_type}")
print(f"Rookie: {card.is_rookie_card}")
print(f"Auto: {card.is_autograph}")
print(f"Relic: {card.is_memorabilia}")
print(f"Grade: {card.grading_company} {card.grade}" if card.grade else "Grade: Raw")
print(f"\nVisual: {card.visual_description}")
print(f"Other: {card.additional_text}")

In [ ]:
# Raw JSON output
print(card.model_dump_json(indent=2))

## 8. eBay Search

In [ ]:
print(f"Search query: {card.ebay_search_query}")
display(Markdown(f"**[🔍 Search eBay for this card]({card.ebay_url})**"))

In [ ]:
# Uncomment to open in browser
# webbrowser.open(card.ebay_url)

## 9. Batch Processing

Process multiple cards at once.

In [ ]:
# Example: batch process multiple cards (front images only for demo)
card_images = [
    {"front": "https://i.ebayimg.com/images/g/fBQAAOSwbRVl8o0d/s-l1600.webp", "back": None},
    # Add more cards here:
    # {"front": "/path/to/card2_front.jpg", "back": "/path/to/card2_back.jpg"},
]

cards = []
for i, img in enumerate(card_images):
    try:
        c = extract_card(front_image=img["front"], back_image=img.get("back"))
        cards.append(c)
        print(f"✓ {i+1}. {c.display_name}")
    except Exception as e:
        print(f"✗ {i+1}. Failed: {e}")

print(f"\nExtracted {len(cards)} card(s)")

In [ ]:
# Convert to DataFrame
if cards:
    df = pd.DataFrame([c.model_dump() for c in cards])
    df["ebay_query"] = [c.ebay_search_query for c in cards]
    df["ebay_url"] = [c.ebay_url for c in cards]
    
    # Display key columns
    display(df[[
        "player_name", "year", "manufacturer", "product_set", 
        "card_number", "serial_number", "parallel_type", 
        "is_rookie_card", "is_autograph"
    ]])

## 10. Tips for Best Results

### Image Quality
- Use well-lit, clear photos
- Capture the entire card (no cropping)
- For graded cards, capture inside the slab

### Front + Back
- **Front**: Parallel type, autograph, serial number, visual appearance
- **Back**: Year (copyright), card number, fine print details

### Parallel Identification
The model knows major parallel types:
- **Topps**: Refractor, Gold, Blue, Green, Purple, Rainbow Foil, X-Fractor, Superfractor
- **Panini**: Silver Prizm, Blue Prizm, Green Wave, Cracked Ice, Snakeskin, Disco, Lazer
- **Upper Deck**: Exclusives, Clear Cut, High Gloss, Outburst

### Visual Description
The `visual_description` field captures colors and effects to help with search:
- "Purple border, holographic shimmer"
- "Cracked ice pattern, rainbow refractor"
- "Stained glass appearance, gold trim"